In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import PIL as pl

import matplotlib.pyplot as plt
import matplotlib.image as mping
import random


from keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping


pathDataset = "C:/Users/Ramiro/Desktop/Inteligencia Artigicial/TPs/IIA-TPS/TP-FINAL/dataset"

numClasses = len(os.listdir(pathDataset + "/train"))
print("Numero de clases: " + str(numClasses))

# Crear generadores con división
trainValDataMod = ImageDataGenerator(rescale=1./255, validation_split=0.2)
testDataMod = ImageDataGenerator(rescale=1./255)

# Datos de entrenamiento (80%)
trainData = trainValDataMod.flow_from_directory(
    directory=pathDataset + "/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="training",  # <-- importante
    shuffle=True
)

# Datos de validación (20%)
valData = trainValDataMod.flow_from_directory(
    directory=pathDataset + "/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="validation",  # <-- importante
    shuffle=True
)

# Datos de prueba
testData = testDataMod.flow_from_directory(
    directory=pathDataset + "/test",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)


# Creao el modelo
baseModel = tf.keras.applications.InceptionV3(include_top = False)

# Freeze
baseModel.trainable = False

# Inputs
inputs = tf.keras.layers.Input(shape = (224, 224, 3), name = "input-layer")

x = baseModel(inputs)
print(f"Forma del modelo luego de recibir los inputs : {x.shape}")

# Pool layer
x = tf.keras.layers.GlobalAveragePooling2D(name = "Global-average-pooling-layer")(x)
print(f"Forma despues del Global Pooling: {x.shape}")

# Ultima layer
outputs = tf.keras.layers.Dense(numClasses, activation = 'softmax', name = 'output-layer')(x)

# Merge
model = tf.keras.Model(inputs, outputs)
model.compile(loss = "categorical_crossentropy",
              optimizer = tf.keras.optimizers.Adam(learning_rate = 0.01),
              metrics = ["accuracy"])

print(model.summary())

Ep = 30
bestModel = "C:/Users/Ramiro/Desktop/Inteligencia Artigicial/TPs/IIA-TPS/TP-FINAL/bestModel.keras"
os.makedirs(os.path.dirname(bestModel), exist_ok=True)

callbacks = [
    ModelCheckpoint(bestModel, verbose = 1, save_best_only = True, monitor = "val_accuracy"),
    ReduceLROnPlateau(monitor = "val_accuracy", patience = 4, factor = 0.1, verbose = 1, min_lr = 1e-6),
    EarlyStopping(monitor = "val_accuracy", patience = 4, verbose = 1)
]


history = model.fit(trainData, 
                    epochs = 30, steps_per_epoch = len(trainData), 
                    validation_data = valData, validation_steps = int(0.25 * len(valData)), callbacks = callbacks)

# Evaluar
print(model.evaluate(testData))

# Plot
def plotLoss(history):
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    accuracy = history.history["accuracy"]
    val_accuracy = history.history["val_accuracy"]

    epochs = range(len(history.history["loss"]))

    plt.plot(epochs, loss, label = "Training loss")
    plt.plot(epochs, val_loss, label = "Val loss")
    plt.title("loss")
    plt.xlabel(epochs)
    plt.legend()
    plt.show()

    plt.plot(epochs, accuracy, label = "Training accuracy")
    plt.plot(epochs, val_accuracy, label = "Val accuracy")
    plt.title("accuracy")
    plt.xlabel(epochs)
    plt.legend()
    plt.show()


plotLoss(history)


Numero de clases: 13
Found 2159 images belonging to 13 classes.
Found 86 images belonging to 13 classes.
Forma del modelo luego de recibir los inputs : (None, 5, 5, 2048)
Forma despues del Global Pooling: (None, 2048)


Model: "functional_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input-layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 5, 5, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Global-average-pooling-layer    │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output-layer (Dense)            │ (None, 13)             │        26,637 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,829,421 (83.27 MB)

 Trainable params: 26,637 (104.05 KB)

 Non-trainable params: 21,802,784 (83.17 MB)

None


c:\Users\Ramiro\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ImportError: Could not import PIL.Image. The use of `load_img` requires PIL.